# Multilingual Keyword Spotting with TinyML Deployment

**Course**: Intro to Deep Learning &nbsp;|&nbsp; **Due**: May 14, 2026

## Overview

We train a compact multilingual wake-word detector on a balanced subset of the
[Multilingual Spoken Words Corpus (MSWC)](https://mlcommons.org/en/multilingual-spoken-words)
(Mazumder et al., NeurIPS 2021). The model simultaneously:

1. **Identifies the spoken language** — 6 languages: English, German, Turkish, Arabic, French, Persian  
2. **Classifies the keyword concept** — silence / unknown / greeting / yes / no

A shared **Depthwise Separable CNN (DS-CNN)** backbone drives both task heads via a
single forward pass. The keyword head is *conditioned* on the language head output,
enabling us to measure **graceful degradation under deliberate language misrouting**
(Section 6) — the core experimental contribution.

**Full pipeline**:  
MSWC streaming subset → 49×40 log-mel spectrograms → multi-task DS-CNN →  
Quantization-Aware Training (QAT) → ONNX export → TFLite (ESP32 target)

---
### Key design choices vs. prior work
| | Harvard few-shot KWS (Mazumder et al., 2021) | This project |
|---|---|---|
| Backbone | EfficientNet-B0 (~11M params) | DS-CNN (<500K params) |
| Training | Pretrain embedding + 5-shot fine-tune | Full joint multi-task training |
| Quantization | Post-training (implied) | QAT from epoch 1 |
| Framework | TensorFlow / TF Lite Micro frontend | PyTorch throughout |

In [ ]:
# Install dependencies not pre-installed on Colab
%pip install -q datasets torchaudio onnx onnxruntime scikit-learn seaborn
%pip install -q tensorflow onnx2tf
# After first run: Runtime > Restart runtime, then skip this cell

In [ ]:
import os, io, json, random, warnings, copy
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from datasets import load_dataset, Audio

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
# ── Language and keyword configuration ────────────────────────────────────────
# Keywords are in native script. MSWC applies a 3-character minimum filter,
# so short words (no/ja/la/نه/لا) may be absent. Run Section 1 (inventory)
# before committing to training to see actual sample counts.

KEYWORD_CONFIG = {
    'en': {
        'name': 'English',
        'greeting': ['hello'],
        'yes':      ['yes'],
        'no':       ['nope'],    # 'no' is 2 chars -> filtered by MSWC
    },
    'de': {
        'name': 'German',
        'greeting': ['hallo'],
        'yes':      ['genau'],   # 'ja' is 2 chars -> filtered; 'genau' = certainly/exactly
        'no':       ['nein'],
    },
    'tr': {
        'name': 'Turkish',
        'greeting': ['merhaba', 'selam'],
        'yes':      ['evet'],
        'no':       ['hayır'],
    },
    'ar': {
        'name': 'Arabic',
        'greeting': ['مرحبا', 'سلام'],
        'yes':      ['نعم'],
        'no':       ['لا'],    # 2 chars -> likely filtered; inventory will confirm
    },
    'fr': {
        'name': 'French',
        'greeting': ['bonjour'],
        'yes':      ['oui'],
        'no':       ['non'],
    },
    'fa': {
        'name': 'Persian',
        'greeting': ['سلام'],
        'yes':      ['بله'],
        'no':       ['نه'],      # 2 chars -> likely filtered; inventory will confirm
    },
}

# Integer labels for both heads
KW_LABELS   = {'silence': 0, 'unknown': 1, 'greeting': 2, 'yes': 3, 'no': 4}
LANG_LABELS = {'en': 0, 'de': 1, 'tr': 2, 'ar': 3, 'fr': 4, 'fa': 5}
KW_NAMES    = {v: k for k, v in KW_LABELS.items()}
LANG_NAMES  = {v: k for k, v in LANG_LABELS.items()}

# ── Audio / spectrogram parameters ────────────────────────────────────────────
# 49 frames x 40 mel bins matches the TF Lite Micro microfrontend standard
# (Mazumder et al.) and makes our results directly comparable to published baselines.
SAMPLE_RATE = 16_000
N_MELS      = 40
N_FFT       = 1024
WIN_LENGTH  = 640   # 40 ms window  (640 samples @ 16kHz)
HOP_LENGTH  = 320   # 20 ms hop     -> (16000-640)/320 + 1 = 49 frames
F_MIN       = 20
F_MAX       = 8_000

# ── Dataset / training hyperparameters ────────────────────────────────────────
SAMPLES_PER_KW_LANG = 200   # cap per (concept, language) for class balance
UNKNOWN_RATIO       = 0.3   # unknown pool size relative to keyword pool
SILENCE_CLIPS       = 400   # white-noise + zero-signal clips
BATCH_SIZE          = 64
EPOCHS              = 30
LR                  = 3e-3
LANG_LOSS_WEIGHT    = 0.5   # lambda: L_total = L_kw + lambda * L_lang

print('Configuration loaded.')
print(f'Languages : {list(KEYWORD_CONFIG.keys())}')
print(f'KW labels : {KW_LABELS}')

---
## Section 1 — Data Inventory

Before downloading anything, we **stream** the MSWC dataset on HuggingFace and count
how many clips exist for each `(language, keyword)` pair. This takes ~2-3 minutes
but saves us from building a pipeline around keywords that don't exist (e.g., short
words filtered by MSWC's 3-character minimum).

The `n_check` parameter controls how many clips to scan per language. Increase it
if counts seem low — Arabic is a low-resource language in MSWC (~7.6h total),
so we may need to scan more clips to find enough target words.

In [ ]:
def inventory_mswc(config: dict, n_check: int = 8000) -> dict:
    """
    Stream the first n_check clips per language and count target keyword hits.
    Returns: {lang_code: {word: count}}
    """
    results = {}
    for lang, cfg in config.items():
        # Collect all target words for this language
        target_words = set()
        for concept, words in cfg.items():
            if isinstance(words, list):
                for w in words:
                    target_words.add(w.lower().strip())

        print(f'\n[{lang}] {cfg["name"]} — scanning {n_check} clips ...')
        counts = defaultdict(int)
        total_scanned = 0
        try:
            ds = load_dataset(
                'MLCommons/ml_spoken_words', lang,
                split='train', streaming=True, trust_remote_code=True
            )
            for sample in ds:
                if total_scanned >= n_check:
                    break
                # Column name may be 'word' or 'keyword' depending on dataset version
                word = sample.get('word', sample.get('keyword', '')).strip().lower()
                if word in target_words:
                    counts[word] += 1
                total_scanned += 1
        except Exception as exc:
            print(f'  ERROR: {exc}')
            results[lang] = {}
            continue

        results[lang] = dict(counts)
        print(f'  Scanned {total_scanned} clips. Hits:')
        for word, cnt in sorted(counts.items(), key=lambda x: -x[1]):
            status = 'OK' if cnt >= 50 else 'LOW' if cnt > 0 else 'MISSING'
            print(f'    [{status}] {word:20s}: {cnt:4d}')
        missing = target_words - set(counts.keys())
        for w in missing:
            print(f'    [MISSING] {w}')

    return results

# Run inventory  (~2-5 min depending on connection speed)
inventory = inventory_mswc(KEYWORD_CONFIG, n_check=8000)
print('\nInventory complete.')

---
## Section 2 — Feature Extraction

### Why log-mel spectrograms?

Raw waveforms are hard to learn from — 1 second at 16 kHz is 16,000 numbers with
complex long-range dependencies. We convert to a 2D time-frequency representation:

**Step 1 — STFT**: Apply a 40 ms Hamming window every 20 ms. The Short-Time Fourier
Transform gives us the complex spectrum per frame. We take the magnitude squared
to get *power*.

**Step 2 — Mel filterbank**: Apply 40 triangular filters whose center frequencies
are spaced on the **Mel scale**:

$$\text{Mel}(f) = 2595 \cdot \log_{10}\!\left(1 + \frac{f}{700}\right)$$

This compresses high frequencies and expands low ones — matching how the human
ear resolves pitch. Two tones 100 Hz apart at 200 Hz sound very different;
the same gap at 5000 Hz is nearly imperceptible.

**Step 3 — Log**: Take `log(energy + ε)`. This compresses the dynamic range
(~60 dB range → ~6 units). Without log, a shout and a whisper of the same
phoneme look completely different to the CNN.

**Why not MFCCs?** MFCCs add a DCT on top of log-mel, keeping only the first
~13 coefficients. The DCT decorrelates filterbank outputs — useful for
GMM-HMM models that need independent features. For CNNs, the convolution
layers *learn* their own decorrelation from the full 40-dim input, so
discarding information via DCT only hurts. Modern KWS papers universally
use log-mel spectrograms.

**Our output**: **(1 × 49 × 40)** per 1-second clip — a grayscale image of the spoken word.

In [ ]:
# ── Build transforms ───────────────────────────────────────────────────────────
mel_transform = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=N_FFT,
    win_length=WIN_LENGTH,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    f_min=F_MIN,
    f_max=F_MAX,
    power=2.0,
).to(device)

amplitude_to_db = T.AmplitudeToDB(stype='power', top_db=80).to(device)

# SpecAugment — applied during training only (Section 3)
spec_augment = nn.Sequential(
    T.FrequencyMasking(freq_mask_param=8),   # mask up to 8 mel bins
    T.TimeMasking(time_mask_param=10),       # mask up to 10 time frames (~200ms)
)

def waveform_to_logmel(waveform: torch.Tensor, sr: int) -> torch.Tensor:
    """
    Convert raw waveform to a normalized log-mel spectrogram.

    Args:
        waveform : (channels, samples) or (samples,) tensor
        sr       : original sample rate
    Returns:
        spec     : (1, 49, 40) tensor  [channel, time_frames, mel_bins]
    """
    # Ensure mono, shape (1, samples)
    if waveform.dim() == 1:
        waveform = waveform.unsqueeze(0)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(0, keepdim=True)

    # Resample to 16 kHz if needed
    if sr != SAMPLE_RATE:
        waveform = T.Resample(sr, SAMPLE_RATE)(waveform)

    # Pad or trim to exactly 1 second
    target_len = SAMPLE_RATE
    if waveform.shape[-1] < target_len:
        waveform = F.pad(waveform, (0, target_len - waveform.shape[-1]))
    else:
        waveform = waveform[..., :target_len]

    waveform = waveform.to(device)
    mel  = mel_transform(waveform)      # (1, 40, 49)  [channel, mel_bins, frames]
    mel  = amplitude_to_db(mel)         # log-compress
    mel  = mel.squeeze(0).T             # (49, 40)     [frames, mel_bins]
    mel  = (mel + 80.0) / 80.0          # normalize to ~[0, 1]
    return mel.unsqueeze(0)             # (1, 49, 40)


# ── Sanity check + visualization ───────────────────────────────────────────────
test_wav = torch.randn(1, SAMPLE_RATE) * 0.1   # white noise placeholder
spec     = waveform_to_logmel(test_wav, SAMPLE_RATE).cpu()

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].imshow(spec.squeeze().numpy().T, aspect='auto', origin='lower', cmap='magma')
axes[0].set_xlabel('Time frame (x20ms)'); axes[0].set_ylabel('Mel bin')
axes[0].set_title(f'Log-mel spectrogram  shape={tuple(spec.shape)}')

# Show what SpecAugment does to a spectrogram
aug_spec = spec_augment(spec.to(device)).cpu()
axes[1].imshow(aug_spec.squeeze().numpy().T, aspect='auto', origin='lower', cmap='magma')
axes[1].set_xlabel('Time frame (x20ms)'); axes[1].set_ylabel('Mel bin')
axes[1].set_title('After SpecAugment (training augmentation)')
plt.tight_layout(); plt.savefig('spectrogram_example.png', dpi=150); plt.show()
print(f'Spectrogram shape: {spec.shape}   (expect torch.Size([1, 49, 40]))')

---
## Section 3 — Dataset

We load MSWC per language in **streaming mode** — no full download required.
For each `(language, concept)` pair we collect up to `SAMPLES_PER_KW_LANG` clips,
then stop streaming. All high-resource languages are *capped at the same count*
to balance classes — essential because Arabic is low-resource (~7.6 h) while
English is high-resource (~1957 h). Without capping, the model would learn
language frequency rather than acoustic features.

We also build:
- **Unknown pool**: random non-target words from each language  
- **Silence pool**: white-noise + zero-signal clips (10% of Speech Commands noise strategy)

Spectrograms are computed on-the-fly in `__getitem__` to avoid caching large tensors.

In [ ]:
def load_language_subset(
    lang: str,
    cfg: dict,
    samples_per_concept: int,
    unknown_per_lang: int = 60,
) -> list:
    """
    Stream MSWC for one language, collect target keywords and unknown words.
    Returns list of dicts: {audio, sr, kw_label, lang_label, word}
    """
    # Build word -> concept mapping (lowercase keys)
    word_to_concept = {}
    for concept, words in cfg.items():
        if concept == 'name' or not isinstance(words, list):
            continue
        for w in words:
            word_to_concept[w.lower().strip()] = concept

    buckets      = defaultdict(list)   # concept -> samples
    unknown_pool = []
    target_concepts = ['greeting', 'yes', 'no']

    try:
        ds = load_dataset(
            'MLCommons/ml_spoken_words', lang,
            split='train', streaming=True, trust_remote_code=True
        )
        # Cast audio column to 16kHz on-the-fly
        ds = ds.cast_column('audio', Audio(sampling_rate=SAMPLE_RATE))
    except Exception as exc:
        print(f'  [WARN] Could not load {lang}: {exc}')
        return []

    def is_full():
        concepts_ok  = all(len(buckets[c]) >= samples_per_concept for c in target_concepts)
        unknown_ok   = len(unknown_pool) >= unknown_per_lang
        return concepts_ok and unknown_ok

    print(f'  Streaming {lang} ({cfg["name"]}) ...')
    for sample in ds:
        if is_full():
            break

        word  = sample.get('word', sample.get('keyword', '')).strip().lower()
        audio = sample['audio']['array'].astype(np.float32)
        sr    = sample['audio']['sampling_rate']

        concept = word_to_concept.get(word)
        if concept and len(buckets[concept]) < samples_per_concept:
            buckets[concept].append({
                'audio': audio, 'sr': sr,
                'kw_label': KW_LABELS[concept],
                'lang_label': LANG_LABELS[lang],
                'word': word,
            })
        elif concept is None and len(unknown_pool) < unknown_per_lang:
            unknown_pool.append({
                'audio': audio, 'sr': sr,
                'kw_label': KW_LABELS['unknown'],
                'lang_label': LANG_LABELS[lang],
                'word': word,
            })

    samples = [s for concept_list in buckets.values() for s in concept_list]
    samples.extend(unknown_pool)
    counts = {c: len(buckets[c]) for c in target_concepts}
    print(f'    {counts}, unknown: {len(unknown_pool)}')
    return samples

In [ ]:
# ── Run data collection ────────────────────────────────────────────────────────
# NOTE: Run inventory (Section 1) first to verify keyword availability.
# This cell streams data and may take 5-15 min depending on connection.

unknown_per_lang = int(SAMPLES_PER_KW_LANG * UNKNOWN_RATIO)
all_samples = []

for lang, cfg in KEYWORD_CONFIG.items():
    lang_samples = load_language_subset(
        lang, cfg,
        samples_per_concept=SAMPLES_PER_KW_LANG,
        unknown_per_lang=unknown_per_lang,
    )
    all_samples.extend(lang_samples)


def make_silence_clips(n: int) -> list:
    """Generate synthetic silence/noise clips for the background class."""
    clips = []
    for _ in range(n // 2):   # white noise at very low amplitude
        clips.append({
            'audio': (np.random.randn(SAMPLE_RATE) * 0.002).astype(np.float32),
            'sr': SAMPLE_RATE, 'kw_label': KW_LABELS['silence'],
            'lang_label': 0, 'word': '__noise__',
        })
    for _ in range(n - n // 2):   # pure silence
        clips.append({
            'audio': np.zeros(SAMPLE_RATE, dtype=np.float32),
            'sr': SAMPLE_RATE, 'kw_label': KW_LABELS['silence'],
            'lang_label': 0, 'word': '__silence__',
        })
    return clips


all_samples.extend(make_silence_clips(SILENCE_CLIPS))
random.shuffle(all_samples)

print(f'\nTotal samples: {len(all_samples)}')
kw_dist   = Counter(s['kw_label']   for s in all_samples)
lang_dist = Counter(s['lang_label'] for s in all_samples)
print('KW dist  :', {KW_NAMES[k]:   v for k, v in sorted(kw_dist.items())})
print('Lang dist:', {LANG_NAMES[k]: v for k, v in sorted(lang_dist.items())})

In [ ]:
# ── Stratified train / val / test split ───────────────────────────────────────
def stratified_split(samples, test_frac=0.15, val_frac=0.15):
    """Split stratified on joint (kw_label, lang_label) to preserve all combinations."""
    strat_labels = [f"{s['kw_label']}_{s['lang_label']}" for s in samples]
    train_val, test = train_test_split(
        samples, test_size=test_frac, stratify=strat_labels, random_state=SEED
    )
    strat_tv = [f"{s['kw_label']}_{s['lang_label']}" for s in train_val]
    train, val = train_test_split(
        train_val,
        test_size=val_frac / (1.0 - test_frac),
        stratify=strat_tv, random_state=SEED
    )
    return train, val, test

train_data, val_data, test_data = stratified_split(all_samples)
print(f'Split sizes — train: {len(train_data)}, val: {len(val_data)}, test: {len(test_data)}')


# ── PyTorch Dataset ────────────────────────────────────────────────────────────
class MSWCDataset(Dataset):
    """
    Converts raw audio dicts to (spectrogram, kw_label, lang_label) triples.
    Augmentation (timeshift + SpecAugment) is applied only during training.
    """
    def __init__(self, samples: list, augment: bool = False):
        self.samples = samples
        self.augment = augment

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        s = self.samples[idx]
        waveform = torch.from_numpy(s['audio'].copy())

        if self.augment:
            # Random timeshift: +/-100ms at 16kHz = +/-1600 samples
            # Simulates imprecise trigger alignment, a common real-world issue.
            shift = random.randint(-1600, 1600)
            waveform = torch.roll(waveform, shift)

        spec = waveform_to_logmel(waveform, s['sr'])   # (1, 49, 40)

        if self.augment:
            spec = spec_augment(spec.to(device)).cpu()

        return (
            spec,
            torch.tensor(s['kw_label'],   dtype=torch.long),
            torch.tensor(s['lang_label'], dtype=torch.long),
        )


def make_loader(data, augment=False, shuffle=True) -> DataLoader:
    ds = MSWCDataset(data, augment=augment)
    return DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=shuffle,
        num_workers=2, pin_memory=(device.type == 'cuda'),
    )

train_loader = make_loader(train_data, augment=True,  shuffle=True)
val_loader   = make_loader(val_data,   augment=False, shuffle=False)
test_loader  = make_loader(test_data,  augment=False, shuffle=False)
print(f'Batches — train: {len(train_loader)}, val: {len(val_loader)}, test: {len(test_loader)}')

---
## Section 4 — Model Architecture

### Depthwise Separable Convolution

A standard Conv2D with kernel $k{\times}k$, $C_{in}$ input and $C_{out}$ output channels
costs $O(k^2 \cdot C_{in} \cdot C_{out})$ multiplications per spatial location.
A **depthwise separable block** splits this into two cheaper operations:

- **Depthwise conv**: $k{\times}k$ filter applied *independently per channel* → $O(k^2 \cdot C_{in})$  
- **Pointwise conv**: $1{\times}1$ conv to mix channels → $O(C_{in} \cdot C_{out})$

Total: $O(k^2 C_{in} + C_{in} C_{out})$ — roughly **8–9× cheaper** than standard conv for $k=3, C=64$.
This is why DS-CNN (Zhang et al., 2017 "Hello Edge") became the standard TinyML KWS backbone.

### Architecture

```
Input: (B, 1, 49, 40)

Backbone:
  Conv2D(1→64, 3×3) + BN + ReLU          → (B,  64, 49, 40)
  DSBlock(64→64)                          → (B,  64, 49, 40)
  DSBlock(64→64)                          → (B,  64, 49, 40)
  MaxPool(2×2)                            → (B,  64, 24, 20)
  DSBlock(64→128, stride=2)               → (B, 128, 12, 10)
  DSBlock(128→128)                        → (B, 128, 12, 10)
  GlobalAvgPool + Flatten                 → (B, 128)

Language head:
  Linear(128 → 6)                         → lang_logits

Keyword head (conditioned on language):
  input = concat(backbone_feat, lang_logits.detach())  → (B, 134)
  Linear(134 → 64) + ReLU + Linear(64 → 5)            → kw_logits
```

**Why condition the keyword head on language?**  
At inference, we can substitute the *wrong* language logits and measure accuracy drop.
If the shared backbone encodes language-agnostic acoustic features well, keyword accuracy
should remain high even under misrouting — this is the core experiment in Section 6.

The `.detach()` on `lang_logits` stops gradients from the keyword loss flowing back
through the language head — each head is trained only by its own loss.

In [ ]:
class DSBlock(nn.Module):
    """Depthwise-separable conv block: DW(3x3)->BN->ReLU -> PW(1x1)->BN->ReLU."""

    def __init__(self, in_ch: int, out_ch: int, stride: int = 1):
        super().__init__()
        self.dw  = nn.Conv2d(in_ch, in_ch, 3, stride=stride, padding=1,
                             groups=in_ch, bias=False)
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.pw  = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.bn1(self.dw(x)))
        x = F.relu(self.bn2(self.pw(x)))
        return x


class MultilingualKWS(nn.Module):
    """
    Shared DS-CNN backbone with two task heads:
      - lang_head  : language identification (n_langs classes)
      - kw_head    : keyword concept (n_kw_classes), conditioned on lang prediction
    """

    def __init__(self, n_langs: int = 6, n_kw_classes: int = 5):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            DSBlock(64, 64),
            DSBlock(64, 64),
            nn.MaxPool2d(2),
            DSBlock(64, 128, stride=2),
            DSBlock(128, 128),
            nn.AdaptiveAvgPool2d(1),
        )
        self.lang_head = nn.Linear(128, n_langs)
        self.kw_head   = nn.Sequential(
            nn.Linear(128 + n_langs, 64),
            nn.ReLU(),
            nn.Linear(64, n_kw_classes),
        )

    def forward(
        self,
        x: torch.Tensor,
        forced_lang_logits: torch.Tensor = None,
    ):
        """
        Args:
            x                  : (B, 1, 49, 40) log-mel spectrogram
            forced_lang_logits : (B, n_langs) optional override for misrouting experiment
        Returns:
            kw_logits   : (B, n_kw_classes)
            lang_logits : (B, n_langs)
        """
        feat        = self.backbone(x).flatten(1)      # (B, 128)
        lang_logits = self.lang_head(feat)             # (B, 6)

        cond = forced_lang_logits if forced_lang_logits is not None else lang_logits.detach()
        kw_logits = self.kw_head(torch.cat([feat, cond], dim=1))   # (B, 5)
        return kw_logits, lang_logits

    def count_params(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


model = MultilingualKWS(
    n_langs=len(LANG_LABELS),
    n_kw_classes=len(KW_LABELS),
).to(device)

print(f'Parameters: {model.count_params():,}')

# Verify shapes with a dummy forward pass
dummy = torch.randn(4, 1, 49, 40).to(device)
kw_out, lang_out = model(dummy)
print(f'KW  logits : {kw_out.shape}    (expect [4, 5])')
print(f'Lang logits: {lang_out.shape}  (expect [4, 6])')

---
## Section 5 — Training

**Loss function**:
$$\mathcal{L} = \mathcal{L}_{\text{kw}} + \lambda \cdot \mathcal{L}_{\text{lang}}$$

Both $\mathcal{L}_{\text{kw}}$ and $\mathcal{L}_{\text{lang}}$ are cross-entropy losses.
$\lambda = 0.5$ makes keyword detection the primary objective while still teaching
the language head.

**Optimizer**: AdamW with cosine LR annealing.  
AdamW decouples weight decay from the gradient update (unlike Adam + L2 regularization),
which tends to generalize better on smaller datasets.

**Early stopping**: we save the checkpoint with the best validation keyword accuracy.

In [ ]:
@torch.no_grad()
def evaluate(loader: DataLoader, model: nn.Module) -> tuple:
    """Returns (kw_accuracy, lang_accuracy) on the given loader."""
    model.eval()
    kw_correct = kw_total = lang_correct = lang_total = 0
    for specs, kw_labels, lang_labels in loader:
        specs, kw_labels, lang_labels = (
            specs.to(device), kw_labels.to(device), lang_labels.to(device)
        )
        kw_logits, lang_logits = model(specs)
        kw_correct   += (kw_logits.argmax(1)   == kw_labels).sum().item()
        lang_correct += (lang_logits.argmax(1) == lang_labels).sum().item()
        kw_total     += kw_labels.size(0)
        lang_total   += lang_labels.size(0)
    return kw_correct / kw_total, lang_correct / lang_total


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = EPOCHS,
    lr: float = LR,
    lam: float = LANG_LOSS_WEIGHT,
    ckpt_path: str = 'best_model.pt',
) -> dict:
    optimizer  = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion  = nn.CrossEntropyLoss()
    history    = {'train_kw': [], 'val_kw': [], 'train_lang': [], 'val_lang': [], 'loss': []}
    best_val_kw = 0.0

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        for specs, kw_labels, lang_labels in train_loader:
            specs, kw_labels, lang_labels = (
                specs.to(device), kw_labels.to(device), lang_labels.to(device)
            )
            kw_logits, lang_logits = model(specs)
            loss = criterion(kw_logits, kw_labels) + lam * criterion(lang_logits, lang_labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        train_kw, train_lang = evaluate(train_loader, model)
        val_kw,   val_lang   = evaluate(val_loader,   model)

        history['train_kw'].append(train_kw)
        history['val_kw'].append(val_kw)
        history['train_lang'].append(train_lang)
        history['val_lang'].append(val_lang)
        history['loss'].append(epoch_loss / len(train_loader))

        if val_kw > best_val_kw:
            best_val_kw = val_kw
            torch.save(model.state_dict(), ckpt_path)

        if epoch % 5 == 0 or epoch == 1:
            print(
                f'Ep {epoch:3d}/{epochs} | loss {history["loss"][-1]:.4f} '
                f'| KW  train {train_kw:.3f}  val {val_kw:.3f} '
                f'| Lang train {train_lang:.3f}  val {val_lang:.3f}'
            )

    # Load best checkpoint
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f'\nTraining complete. Best val KW accuracy: {best_val_kw:.3f}')
    return history


history = train_model(model, train_loader, val_loader)

In [ ]:
# ── Training curves ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
eps = range(1, len(history['loss']) + 1)

axes[0].plot(eps, history['loss'])
axes[0].set_title('Training Loss'); axes[0].set_xlabel('Epoch')

axes[1].plot(eps, history['train_kw'], label='Train')
axes[1].plot(eps, history['val_kw'],   label='Val')
axes[1].set_title('Keyword Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(eps, history['train_lang'], label='Train')
axes[2].plot(eps, history['val_lang'],   label='Val')
axes[2].set_title('Language Accuracy'); axes[2].set_xlabel('Epoch'); axes[2].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

---
## Section 6 — Evaluation & Misrouting Experiment

### Per-class evaluation

We report precision / recall / F1 for both heads on the held-out test set.

### Misrouting experiment

This is the core contribution of our architecture. At inference, we force the keyword
head to receive the logits of the *wrong* language and measure how much keyword accuracy
degrades. The results matrix has:
- **Diagonal**: correct routing (baseline accuracy per language)
- **Off-diagonal**: accuracy when routed to a different language

**Hypothesis**: since the shared backbone encodes *acoustic phonetic content* rather than
language identity, keyword accuracy should remain relatively high even when misrouted.
Acoustically similar pairs (merhaba/marhaba, hello/hallo) should suffer the least;
phonetically distinct pairs (bonjour/merhaba) may show larger drops.

In [ ]:
@torch.no_grad()
def full_eval(loader: DataLoader, model: nn.Module) -> tuple:
    model.eval()
    kw_pred, kw_true, lang_pred, lang_true = [], [], [], []
    for specs, kw_labels, lang_labels in loader:
        specs = specs.to(device)
        kw_logits, lang_logits = model(specs)
        kw_pred.extend(kw_logits.argmax(1).cpu().tolist())
        kw_true.extend(kw_labels.tolist())
        lang_pred.extend(lang_logits.argmax(1).cpu().tolist())
        lang_true.extend(lang_labels.tolist())
    return (
        np.array(kw_pred), np.array(kw_true),
        np.array(lang_pred), np.array(lang_true),
    )


kw_pred, kw_true, lang_pred, lang_true = full_eval(test_loader, model)

kw_names_list   = [KW_NAMES[i]   for i in sorted(KW_NAMES)]
lang_names_list = [LANG_NAMES[i] for i in sorted(LANG_NAMES)]

print('=== Keyword Classification (test set) ===')
print(classification_report(kw_true, kw_pred, target_names=kw_names_list))
print('=== Language Identification (test set) ===')
print(classification_report(lang_true, lang_pred, target_names=lang_names_list))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (pred, true, names, title) in zip(axes, [
    (kw_pred, kw_true, kw_names_list, 'Keyword confusion matrix'),
    (lang_pred, lang_true, lang_names_list, 'Language confusion matrix'),
]):
    cm = confusion_matrix(true, pred, normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2f', xticklabels=names, yticklabels=names,
                cmap='Blues', ax=ax)
    ax.set_title(title); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150)
plt.show()

In [ ]:
@torch.no_grad()
def misrouting_experiment(
    test_samples: list,
    model: nn.Module,
    n_per_lang: int = 80,
) -> np.ndarray:
    """
    For each (true_lang, forced_lang) pair, compute keyword accuracy when the
    keyword head receives logits for forced_lang instead of the true language.

    Returns: accuracy matrix  [true_lang x forced_lang], shape (n_langs, n_langs)
    """
    model.eval()
    lang_ids = sorted(LANG_LABELS.values())
    n_langs  = len(lang_ids)
    results  = np.zeros((n_langs, n_langs))

    # Separate test samples by language (exclude silence/unknown for clean measurement)
    lang_pools = defaultdict(list)
    for s in test_samples:
        if s['kw_label'] not in (KW_LABELS['silence'], KW_LABELS['unknown']):
            lang_pools[s['lang_label']].append(s)

    for true_lang in lang_ids:
        pool = lang_pools[true_lang][:n_per_lang]
        if not pool:
            continue
        loader = DataLoader(MSWCDataset(pool, augment=False),
                            batch_size=32, shuffle=False)

        for forced_lang in lang_ids:
            correct = total = 0
            for specs, kw_labels, _ in loader:
                specs, kw_labels = specs.to(device), kw_labels.to(device)
                B = specs.size(0)

                # Construct a near-one-hot forced language signal
                forced = torch.zeros(B, n_langs, device=device)
                forced[:, forced_lang] = 100.0   # dominates after softmax

                kw_logits, _ = model(specs, forced_lang_logits=forced)
                correct += (kw_logits.argmax(1) == kw_labels).sum().item()
                total   += kw_labels.size(0)

            results[true_lang][forced_lang] = correct / max(total, 1)

    # Plot
    lang_list = [LANG_NAMES[i] for i in lang_ids]
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        results, annot=True, fmt='.2f',
        xticklabels=lang_list, yticklabels=lang_list,
        cmap='RdYlGn', vmin=0.0, vmax=1.0,
    )
    plt.xlabel('Forced routing language')
    plt.ylabel('True language')
    plt.title('Keyword accuracy under deliberate language misrouting\n(diagonal = correct routing)')
    plt.tight_layout()
    plt.savefig('misrouting_experiment.png', dpi=150)
    plt.show()
    return results


misrouting_matrix = misrouting_experiment(test_data, model)

# Summary statistics
diag      = np.diag(misrouting_matrix)
off_diag  = misrouting_matrix[~np.eye(len(diag), dtype=bool)]
print(f'Mean correct-routing accuracy : {diag.mean():.3f}')
print(f'Mean misrouted accuracy       : {off_diag.mean():.3f}')
print(f'Accuracy degradation          : {diag.mean() - off_diag.mean():.3f}')

---
## Section 7 — Quantization-Aware Training (QAT)

A float32 model stores each weight as 4 bytes. Our ~400K parameter model therefore
needs ~1.6 MB — too large for the ESP32's 512 KB SRAM. **8-bit quantization** maps
weights from float32 to `int8`, reducing size ~4×.

**Post-training quantization (PTQ)**: convert after training. Fast but can drop 1–3%
accuracy because the model was never exposed to rounding errors during training.

**Quantization-Aware Training (QAT)**: insert *fake quantization nodes* (round-to-nearest
with straight-through estimator for gradients) throughout the graph *during training*.
The model learns weight distributions that survive quantization. Typical accuracy loss:
<0.5% vs float32.

We use `torch.ao.quantization` with the **QNNPACK backend** (ARM-optimized integer
math — the same instruction set family as the ESP32's Xtensa LX7 cores).

In [ ]:
import copy
from torch.ao.quantization import get_default_qat_qconfig, prepare_qat, convert

torch.backends.quantized.engine = 'qnnpack'

# Clone the trained float model before inserting quantization stubs
qat_model = copy.deepcopy(model).cpu()
qat_model.train()

qconfig = get_default_qat_qconfig('qnnpack')
qat_model.qconfig = qconfig
prepare_qat(qat_model, inplace=True)
qat_model = qat_model.to(device)

print('QAT model prepared — fake quantization nodes inserted.')
print(f'Parameters: {sum(p.numel() for p in qat_model.parameters()):,}')

# Fine-tune with QAT for a short phase at lower LR
QAT_EPOCHS = 10
QAT_LR     = 5e-5

qat_optimizer = torch.optim.AdamW(qat_model.parameters(), lr=QAT_LR, weight_decay=1e-4)
qat_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(qat_optimizer, T_max=QAT_EPOCHS)
criterion     = nn.CrossEntropyLoss()
best_qat_kw   = 0.0

print(f'\nQAT fine-tuning ({QAT_EPOCHS} epochs at lr={QAT_LR}) ...')
for epoch in range(1, QAT_EPOCHS + 1):
    qat_model.train()
    for specs, kw_labels, lang_labels in train_loader:
        specs, kw_labels, lang_labels = (
            specs.to(device), kw_labels.to(device), lang_labels.to(device)
        )
        kw_logits, lang_logits = qat_model(specs)
        loss = (criterion(kw_logits, kw_labels) +
                LANG_LOSS_WEIGHT * criterion(lang_logits, lang_labels))
        qat_optimizer.zero_grad()
        loss.backward()
        qat_optimizer.step()
    qat_scheduler.step()

    val_kw, val_lang = evaluate(val_loader, qat_model)
    if val_kw > best_qat_kw:
        best_qat_kw = val_kw
        torch.save(qat_model.state_dict(), 'best_qat_model.pt')
    print(f'  QAT ep {epoch:2d}/{QAT_EPOCHS} | KW val {val_kw:.3f} | Lang val {val_lang:.3f}')

# Convert: replace fake quant nodes with real int8 ops
qat_model.load_state_dict(torch.load('best_qat_model.pt', map_location='cpu'))
qat_model.eval().cpu()
quantized_model = convert(copy.deepcopy(qat_model), inplace=False)
print('\nConverted to int8 quantized model.')

# Compare sizes
def model_size_mb(m: nn.Module) -> float:
    buf = io.BytesIO()
    torch.save(m.state_dict(), buf)
    return buf.tell() / 1e6

fp32_mb = model_size_mb(model.cpu())
int8_mb = model_size_mb(quantized_model)
print(f'Float32 model : {fp32_mb:.2f} MB')
print(f'Int8 model    : {int8_mb:.2f} MB  ({fp32_mb / int8_mb:.1f}x smaller)')

# Move float model back to device for remaining cells
model = model.to(device)

---
## Section 8 — Export

### Export path

```
PyTorch (float32) ──▶ ONNX ──▶ TFLite (.tflite)
                                    │
                                    └─▶ TFLite Micro (ESP32)
```

**ONNX** (Open Neural Network Exchange) is a portable computational graph format
understood by most inference frameworks. We validate the exported graph with ONNX Runtime
before proceeding to TFLite.

**TFLite / TFLite Micro** is the embedded deployment target. TFLite Micro supports
ESP32 via the Xtensa toolchain and has known-good support for the Conv2D,
DepthwiseConv2D, and FullyConnected operations used in our model.

We export the **float32 model** (for validation) and note that the int8 quantized
model follows the same path — TFLite's `Optimize.DEFAULT` flag applies dynamic
range quantization if the int8 model is not passed directly.

In [ ]:
import onnx
import onnxruntime as ort

model.eval().cpu()
dummy_input = torch.randn(1, 1, 49, 40)

torch.onnx.export(
    model,
    dummy_input,
    'multilingual_kws_fp32.onnx',
    input_names=['spectrogram'],
    output_names=['kw_logits', 'lang_logits'],
    dynamic_axes={'spectrogram': {0: 'batch_size'}},
    opset_version=17,
    verbose=False,
)

# Verify the exported graph
onnx_model = onnx.load('multilingual_kws_fp32.onnx')
onnx.checker.check_model(onnx_model)
print('ONNX graph validated.')

# Cross-check: PyTorch vs ONNX Runtime outputs should match
ort_sess   = ort.InferenceSession('multilingual_kws_fp32.onnx')
ort_inputs = {'spectrogram': dummy_input.numpy()}
ort_kw, ort_lang = ort_sess.run(None, ort_inputs)

with torch.no_grad():
    pt_kw, pt_lang = model(dummy_input)

max_diff_kw   = np.abs(ort_kw   - pt_kw.numpy()).max()
max_diff_lang = np.abs(ort_lang - pt_lang.numpy()).max()
print(f'Max KW  output diff  (PT vs ORT): {max_diff_kw:.2e}   (expect < 1e-4)')
print(f'Max Lang output diff (PT vs ORT): {max_diff_lang:.2e}   (expect < 1e-4)')

print(f'\nONNX file size: {os.path.getsize("multilingual_kws_fp32.onnx") / 1e3:.1f} KB')
model = model.to(device)  # restore device

In [ ]:
# ── ONNX -> TFLite conversion ──────────────────────────────────────────────────
# Uses onnx2tf (pip install onnx2tf) which is more reliable than onnx-tf.
# If this cell fails, the ONNX model above is still valid for deployment
# via ONNX Runtime for Microcontrollers (ONNX Runtime Mobile).

try:
    import onnx2tf
    onnx2tf.convert(
        input_onnx_file_path='multilingual_kws_fp32.onnx',
        output_folder_path='tflite_output',
        non_verbose=True,
    )
    # The .tflite file will be at tflite_output/multilingual_kws_fp32.tflite
    import glob
    tflite_files = glob.glob('tflite_output/*.tflite')
    if tflite_files:
        tflite_path = tflite_files[0]
        size_kb = os.path.getsize(tflite_path) / 1024
        print(f'TFLite model: {tflite_path}  ({size_kb:.1f} KB)')
        print('Compatible with TFLite Micro on ESP32 (Xtensa LX7).')

        # Quick TFLite inference check
        import tensorflow as tf
        interp = tf.lite.Interpreter(model_path=tflite_path)
        interp.allocate_tensors()
        inp_det = interp.get_input_details()[0]
        out_det = interp.get_output_details()
        interp.set_tensor(inp_det['index'], dummy_input.numpy())
        interp.invoke()
        tflite_kw = interp.get_tensor(out_det[0]['index'])
        print(f'TFLite inference output shape: {tflite_kw.shape}   (expect (1, 5))')
    else:
        print('No .tflite file found in output folder — check onnx2tf output above.')

except ImportError:
    print('onnx2tf not installed. Run: pip install onnx2tf')
    print('Alternatively, convert the ONNX model at: https://netron.app (visualization)')
    print('Or use: https://www.tensorflow.org/lite/guide/ops_compatibility')
except Exception as exc:
    print(f'TFLite conversion failed: {exc}')
    print('The ONNX model (multilingual_kws_fp32.onnx) is ready for alternative runtimes.')

In [ ]:
# ── Save full checkpoint ───────────────────────────────────────────────────────
torch.save({
    'model_state_dict': model.state_dict(),
    'history': history,
    'config': {
        'KEYWORD_CONFIG':     KEYWORD_CONFIG,
        'KW_LABELS':          KW_LABELS,
        'LANG_LABELS':        LANG_LABELS,
        'SAMPLE_RATE':        SAMPLE_RATE,
        'N_MELS':             N_MELS,
        'N_FFT':              N_FFT,
        'WIN_LENGTH':         WIN_LENGTH,
        'HOP_LENGTH':         HOP_LENGTH,
        'SAMPLES_PER_KW_LANG': SAMPLES_PER_KW_LANG,
    },
}, 'multilingual_kws_checkpoint.pt')
print('Checkpoint saved: multilingual_kws_checkpoint.pt')

# Summary of all saved artefacts
artefacts = [
    'best_model.pt',
    'best_qat_model.pt',
    'multilingual_kws_checkpoint.pt',
    'multilingual_kws_fp32.onnx',
    'spectrogram_example.png',
    'training_curves.png',
    'confusion_matrices.png',
    'misrouting_experiment.png',
]
print('\nArtefacts:')
for f in artefacts:
    if os.path.exists(f):
        print(f'  {f:45s} {os.path.getsize(f)/1e3:8.1f} KB')
    else:
        print(f'  {f:45s}  (not yet generated)')